# Experiment: classical ML second-stage filter for false positives

Both Approach 1 and Approach 2 share a common failure mode: the detector confuses
other metal hardware (screws, plates) for real osteotomy sites, giving a 43-47%
false-positive rate on the test set. This notebook tests an idea for addressing it
without touching the detector itself: run a small classical ML classifier (Random
Forest) on top of each of Approach 1's predicted boxes, trained to tell "real join"
apart from "false positive" using simple numeric features (not a second neural
network) extracted from the box's image crop, then use it to filter out predictions
it's confident are wrong.

**Method.** The detector (Approach 1, `osteotomy_yolov8n`) is run on the **train
split** to collect its own real predictions (including its own false positives) -
each predicted box is labelled 1 (real) if it matches a ground-truth box at IoU>=0.5,
else 0 (false positive). For each box, a 16x16 grayscale crop (with small padding for
context) is taken from the **original full-resolution** image and flattened to 256
raw pixel values, plus 8 extra numeric features (crop mean/std/min/max brightness,
box width, height, aspect ratio, and the detector's own confidence score) - 264
numbers per box. A Random Forest is trained on these to classify real vs. false
positive, then applied to the detector's predictions on the **test split** (never
seen by the filter) to see whether it can suppress the detector's actual false
positives.

In [1]:
import json
from pathlib import Path

import numpy as np
from PIL import Image
from ultralytics import YOLO
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

DATASET_DIR = Path("../dataset")
IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"
SPLITS = json.load(open(DATASET_DIR / "splits.json"))
MODEL_PATH = Path("runs/runs/osteotomy_yolov8n/weights/best.pt")  # Approach 1 (resize)

CROP_SIZE = 16
PAD_FRAC = 0.2

model = YOLO(str(MODEL_PATH))


def load_gt_boxes(stem, w, h):
    label_path = LABELS_DIR / f"{stem}.txt"
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        _, xc, yc, bw, bh = [float(x) for x in line.split()]
        boxes.append([(xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h])
    return boxes


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    union = area_a+area_b-inter
    return inter/union if union > 0 else 0.0


def extract_features(img, box, conf):
    x1, y1, x2, y2 = box
    w, h = x2-x1, y2-y1
    px, py = w*PAD_FRAC, h*PAD_FRAC
    cx1, cy1 = max(0, x1-px), max(0, y1-py)
    cx2, cy2 = min(img.width, x2+px), min(img.height, y2+py)
    crop = img.crop((cx1, cy1, cx2, cy2)).convert("L").resize((CROP_SIZE, CROP_SIZE))
    arr = np.asarray(crop, dtype=np.float32) / 255.0
    pixel_feats = arr.flatten()
    stat_feats = np.array([arr.mean(), arr.std(), arr.min(), arr.max(),
                            w, h, w/h if h > 0 else 0, conf], dtype=np.float32)
    return np.concatenate([pixel_feats, stat_feats])


def collect_predictions(patient_ids, conf_thresh=0.25):
    rows = []
    n_gt_total = 0
    for img_path in sorted(IMAGES_DIR.glob("*.jpg")):
        pid = img_path.stem.split("_")[0]
        if pid not in patient_ids:
            continue
        img = Image.open(img_path)
        w, h = img.size
        gt_boxes = load_gt_boxes(img_path.stem, w, h)
        n_gt_total += len(gt_boxes)

        pred = model.predict(str(img_path), imgsz=320, conf=conf_thresh, verbose=False)[0]
        pred_boxes = pred.boxes.xyxy.cpu().numpy().tolist() if len(pred.boxes) else []
        pred_confs = pred.boxes.conf.cpu().numpy().tolist() if len(pred.boxes) else []

        matched_gt = set()
        for pb, conf in zip(pred_boxes, pred_confs):
            best_iou, best_gi = 0, -1
            for gi, gb in enumerate(gt_boxes):
                if gi in matched_gt:
                    continue
                v = iou(pb, gb)
                if v > best_iou:
                    best_iou, best_gi = v, gi
            label = 1 if best_iou >= 0.5 else 0
            if label == 1:
                matched_gt.add(best_gi)
            feats = extract_features(img, pb, conf)
            rows.append({"stem": img_path.stem, "box": pb, "label": label, "features": feats})
    return rows, n_gt_total


train_patients = {p for p, s in SPLITS.items() if s == "train"}
test_patients = {p for p, s in SPLITS.items() if s == "test"}
print("train patients:", len(train_patients), "| test patients:", len(test_patients))

train patients: 35 | test patients: 25


Collect the detector's real predictions on the train split, and label each one TP/FP against ground truth.

In [2]:
train_rows, _ = collect_predictions(train_patients)
X_train = np.stack([r["features"] for r in train_rows])
y_train = np.array([r["label"] for r in train_rows])
print(f"Train candidates: {len(y_train)} (real={y_train.sum()}, false_positive={(y_train==0).sum()})")

Train candidates: 828 (real=685, false_positive=143)


Train the Random Forest filter.

In [3]:
clf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
print("Trained.")

Trained.


Apply the same detector + labelling to the **test split**, then test the filter on its predictions.

In [4]:
test_rows, n_gt_test = collect_predictions(test_patients)
X_test = np.stack([r["features"] for r in test_rows])
y_test = np.array([r["label"] for r in test_rows])
print(f"Test candidates: {len(y_test)} (real={y_test.sum()}, false_positive={(y_test==0).sum()})")
print(f"Total GT boxes in test split: {n_gt_test}")

tp_base = int(y_test.sum())
fp_base = int((y_test == 0).sum())
precision_base = tp_base / (tp_base + fp_base)
recall_base = tp_base / n_gt_test
print(f"\n=== Baseline (YOLO alone, conf=0.25) ===")
print(f"TP={tp_base}, FP={fp_base}, precision={precision_base:.3f}, recall={recall_base:.3f}")

keep_pred = clf.predict(X_test)
tp_filt = int(((y_test == 1) & (keep_pred == 1)).sum())
fp_filt = int(((y_test == 0) & (keep_pred == 1)).sum())
tp_removed_by_mistake = int(((y_test == 1) & (keep_pred == 0)).sum())
fp_correctly_removed = int(((y_test == 0) & (keep_pred == 0)).sum())
precision_filt = tp_filt / (tp_filt + fp_filt) if (tp_filt+fp_filt) > 0 else 0
recall_filt = tp_filt / n_gt_test
print(f"\n=== After Random Forest filter ===")
print(f"TP={tp_filt}, FP={fp_filt}, precision={precision_filt:.3f}, recall={recall_filt:.3f}")
print(f"False positives correctly removed: {fp_correctly_removed}/{fp_base} ({fp_correctly_removed/fp_base:.1%})")
print(f"True positives incorrectly removed: {tp_removed_by_mistake}/{tp_base} ({tp_removed_by_mistake/tp_base:.1%})")

print("\n=== Filter's own classification accuracy on test candidates ===")
print(classification_report(y_test, keep_pred, target_names=["false_positive", "real_join"]))

Test candidates: 166 (real=94, false_positive=72)
Total GT boxes in test split: 173

=== Baseline (YOLO alone, conf=0.25) ===
TP=94, FP=72, precision=0.566, recall=0.543

=== After Random Forest filter ===
TP=92, FP=68, precision=0.575, recall=0.532
False positives correctly removed: 4/72 (5.6%)
True positives incorrectly removed: 2/94 (2.1%)

=== Filter's own classification accuracy on test candidates ===
                precision    recall  f1-score   support

false_positive       0.67      0.06      0.10        72
     real_join       0.57      0.98      0.72        94

      accuracy                           0.58       166
     macro avg       0.62      0.52      0.41       166
  weighted avg       0.61      0.58      0.45       166



Which features did the filter actually rely on?

In [5]:
feat_names = [f"px_{i}" for i in range(CROP_SIZE*CROP_SIZE)] + ["mean", "std", "min", "max", "w", "h", "aspect", "yolo_conf"]
importances = clf.feature_importances_
stat_importances = sorted(zip(feat_names[-8:], importances[-8:]), key=lambda x: -x[1])
pixel_importance_total = importances[:-8].sum()

print("Stat-feature importances:")
for name, imp in stat_importances:
    print(f"  {name}: {imp:.4f}")
print(f"Combined importance of all {CROP_SIZE*CROP_SIZE} pixel features: {pixel_importance_total:.4f}")
print(f"(i.e. average per-pixel-feature importance: {pixel_importance_total/(CROP_SIZE*CROP_SIZE):.5f})")

Stat-feature importances:
  yolo_conf: 0.1242
  w: 0.0061
  h: 0.0045
  mean: 0.0039
  std: 0.0038
  aspect: 0.0030
  min: 0.0028
  max: 0.0025
Combined importance of all 256 pixel features: 0.8492
(i.e. average per-pixel-feature importance: 0.00332)


## Simpler comparison: just raise the confidence threshold

The filter leaned heavily on the detector's own confidence score rather than the
pixel content. That raises the question of whether a Random Forest was needed at
all - or whether simply requiring a higher confidence score from the detector
achieves the same effect, more simply.

In [6]:
for conf in [0.25, 0.4, 0.5, 0.6]:
    m = model.val(data="../dataset/yolo/data.yaml", split="test", imgsz=320, conf=conf, plots=False, verbose=False)
    print(f"conf={conf}: precision={m.box.mp:.3f}, recall={m.box.mr:.3f}")

Ultralytics 8.4.115  Python-3.11.9 torch-2.13.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)


val: Fast image access  (ping: 0.00.0 ms, read: 445.3384.7 MB/s, size: 37.2 KB)


val: Scanning E:\Bone Union Detection\dataset\yolo\labels\test.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123  0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 12% ━─────────── 1/8 1.0s/it 0.3s<7.3s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 25% ━━━───────── 2/8 1.6it/s 0.6s<3.7s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 87% ━━━━━━━━━━── 7/8 3.3it/s 2.0s<0.3s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.7it/s 2.2s

                   all        123        173       0.73      0.266      0.218     0.0665


Speed: 0.2ms preprocess, 15.2ms inference, 0.0ms loss, 0.1ms postprocess per image


conf=0.6: precision=0.730, recall=0.266


## Conclusion

**The Random Forest filter did not meaningfully work.** It removed only 4 of 72 test
false positives (5.6%) while incorrectly discarding 2 real detections, moving
precision from 0.566 to just 0.575 - barely different from baseline, and its own
classification accuracy (58%) was barely above the 57% "always predict real"
majority-class baseline. The feature importances explain why: individual pixel
features carried almost no weight, while the detector's own confidence score
dominated - i.e. the filter had learned little from image content beyond what the
detector's confidence already encoded.

**Simple confidence-threshold tuning achieves a much bigger, cleaner effect**: raising
the threshold from 0.25 to 0.5 takes precision from 0.57 to 0.72 (a real improvement,
vs. the filter's negligible 0.566->0.575), at the honest cost of recall dropping from
0.55 to 0.42 - a genuine, explainable precision/recall trade-off via one number,
instead of a whole extra model that didn't add discriminative power.

**Why the classifier likely failed**: the deepest reason is probably that a real join
and a confusable screw genuinely look similar in raw pixel intensity/brightness terms
(both are small bright blobs near bone) - the actual distinguishing signal is more
subtle (bone texture patterns, precise shape) than simple flattened pixel values and
basic brightness statistics can capture. The YOLO detector itself, with a full
convolutional architecture and far more parameters, already struggles with exactly
this distinction; a much smaller classical model on cruder features was unlikely to
do better. A more promising version of this idea (left as future work) would reuse
the CNN's own learned features (e.g. its penultimate-layer embedding) as the
classifier's input instead of raw pixels - richer signal than hand-crafted features,
without needing to train a whole second neural network from scratch.